# ANFIS Predict — Toàn bộ tập WBCD

Notebook load model đã huấn luyện (stamp trong cell cấu hình) và dự đoán trên **toàn bộ** Wisconsin Breast Cancer Dataset (UCI original, sau khi loại mẫu thiếu `bare_nuclei`).

Pipeline inference bám đúng lúc train:
1. Chuẩn hóa 9 đặc trưng bằng `scaler.pkl` đã lưu
2. Chọn 3 feature cố định theo paper (v1, v2, v3)
3. Load ANFIS checkpoint + predict
4. Đánh giá metric và lưu kết quả dự đoán

In [20]:
import warnings
warnings.filterwarnings("ignore")

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from skanfis import scikit_anfis
from skanfis.fs import FS, LinguisticVariable, GaussianFuzzySet

print("Torch:", torch.__version__)

Torch: 2.12.0+cpu


In [21]:
# Cau hinh artifact cua lan train 20260619_090230
STAMP = "20260621_174830"
RUN_TAG = "paperstyle_pca_feature_select"
models_dir = Path("models")

model_path = models_dir / f"{STAMP}_{RUN_TAG}_best_model.pkl"
scaler_path = models_dir / f"{STAMP}_paperstyle_scaler.pkl"
meta_path = models_dir / f"{STAMP}_paperstyle_meta.json"
membership_csv_path = models_dir / f"{STAMP}_paperstyle_membership_functions.csv"
metrics_json_path = models_dir / f"{STAMP}_paperstyle_metrics.json"
quality_drop_csv_path = models_dir / f"{STAMP}_paperstyle_quality_dropped_samples.csv"

for p in [model_path, scaler_path, meta_path, membership_csv_path]:
    if not p.exists():
        raise FileNotFoundError(f"Thieu file: {p}")

with open(meta_path, encoding="utf-8") as f:
    meta = json.load(f)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

with open(metrics_json_path, encoding="utf-8") as f:
    saved_metrics = json.load(f)

selected_feature_names = meta["selected_features"]
print("Model:", model_path)
print("Selected features:", selected_feature_names)
print("Train meta — used samples (paper split):", meta["used_samples"])

Model: models\20260621_174830_paperstyle_pca_feature_select_best_model.pkl
Selected features: ['clump_thickness', 'uniformity_of_cell_size', 'uniformity_of_cell_shape']
Train meta — used samples (paper split): 663


In [22]:
# Load WBCD goc (UCI Breast Cancer Wisconsin Original)
uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    "sample_code_number",
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(uci_url, header=None, names=cols)
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)
df["target"] = (df["class"] == 4).astype(int)
df["target_label"] = df["target"].map({0: "benign", 1: "malignant"})

feature_cols = [
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
]

X_raw = df[feature_cols].values.astype(np.float32)
y_true = df["target"].values.astype(int)

print("WBCD sau loai missing:", X_raw.shape)
print("Class balance (malignant rate):", round(float(y_true.mean()), 4))

WBCD sau loai missing: (683, 9)
Class balance (malignant rate): 0.3499


In [23]:
# Tien xu ly: scaler da fit luc train + chon 3 feature paper
X_norm = scaler.transform(X_raw).astype(np.float32)
selected_idx = [feature_cols.index(c) for c in selected_feature_names]
X_model = X_norm[:, selected_idx].astype(np.float32)

print("Input shape cho ANFIS:", X_model.shape)

Input shape cho ANFIS: (683, 3)


In [24]:
# Khoi tao kien truc ANFIS giong luc train (FS + ten bien ngan), roi load checkpoint
feature_name_map = {
    "clump_thickness": "Clump Thickness",
    "uniformity_of_cell_size": "Uniformity of Cell Size",
    "uniformity_of_cell_shape": "Uniformity of Cell Shape",
}
FS_VAR_NAMES = {
    "clump_thickness": "ClumpThickness",
    "uniformity_of_cell_size": "CellSize",
    "uniformity_of_cell_shape": "CellShape",
}
FS_DISPLAY_NAMES = {v: feature_name_map[k] for k, v in FS_VAR_NAMES.items()}
LINGUISTIC_TERMS = ("low", "medium", "high")
mf_init_by_display = {item["var"]: item for item in meta["mf_init_log"]}

fs = FS()
for feat_col in selected_feature_names:
    fs_var = FS_VAR_NAMES[feat_col]
    item = mf_init_by_display[feature_name_map[feat_col]]
    centers = item["mu"]
    sigma = float(item["sigma"])
    mf_low = GaussianFuzzySet(mu=centers[0], sigma=sigma, term="low")
    mf_med = GaussianFuzzySet(mu=centers[1], sigma=sigma, term="medium")
    mf_high = GaussianFuzzySet(mu=centers[2], sigma=sigma, term="high")
    fs.add_linguistic_variable(
        fs_var,
        LinguisticVariable([mf_low, mf_med, mf_high], concept=FS_DISPLAY_NAMES[fs_var]),
    )

fs.set_crisp_output_value("benign", 0)
fs.set_crisp_output_value("malignant", 1)
fs.add_rules(meta["expert_rules"])

model = scikit_anfis(
    fs,
    description="WBCD_PaperStyle_ANFIS_Inference",
    epoch=meta.get("epoch", 300),
    hybrid=True,
    label="c",
    zerotype=True,
)
model.load(str(model_path))
model.eval()
model.is_training = False

print("Loaded model:", model_path.name)
print("Num rules:", model.num_rules)

 * Detected Sugeno model type
Loaded model: 20260621_174830_paperstyle_pca_feature_select_best_model.pkl
Num rules: 27


In [25]:
def to_binary(pred):
    pred = np.asarray(pred).reshape(-1)
    return np.clip(np.round(pred), 0, 1).astype(int)


def evaluate_split(name, y_true_arr, y_pred_arr):
    acc = accuracy_score(y_true_arr, y_pred_arr)
    prec = precision_score(y_true_arr, y_pred_arr, zero_division=0)
    rec = recall_score(y_true_arr, y_pred_arr, zero_division=0)
    f1 = f1_score(y_true_arr, y_pred_arr, zero_division=0)
    try:
        auc = roc_auc_score(y_true_arr, y_pred_arr)
    except ValueError:
        auc = np.nan

    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "split": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": None if np.isnan(auc) else float(auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support": int(len(y_true_arr)),
    }


# Predict toan bo WBCD
y_pred_raw = model.predict(X_model)
y_pred = to_binary(y_pred_raw)

result_df = df[["sample_code_number", "class", "target", "target_label"]].copy()
result_df["y_pred_raw"] = np.asarray(y_pred_raw).reshape(-1)
result_df["y_pred"] = y_pred
result_df["pred_label"] = result_df["y_pred"].map({0: "benign", 1: "malignant"})
result_df["correct"] = (result_df["target"] == result_df["y_pred"]).astype(int)

full_metrics = evaluate_split("FULL_WBCD", y_true, y_pred)

print("\n=== Metrics tren TOAN BO WBCD (683 mau) ===")
for k, v in full_metrics.items():
    if k != "split":
        print(f"- {k}: {v}")

print("\nConfusion matrix (FULL):")
print(confusion_matrix(y_true, y_pred))

print("\n5 mau dau:")
display(result_df.head())

print("\n5 mau du doan sai:")
display(result_df[result_df["correct"] == 0].head())


=== Metrics tren TOAN BO WBCD (683 mau) ===
- accuracy: 0.9516837481698389
- precision: 0.9221311475409836
- recall: 0.9414225941422594
- f1_score: 0.9316770186335404
- roc_auc: 0.9493149006747333
- tn: 425
- fp: 19
- fn: 14
- tp: 225
- support: 683

Confusion matrix (FULL):
[[425  19]
 [ 14 225]]

5 mau dau:


,sample_code_number,class,target,target_label,y_pred_raw,y_pred,pred_label,correct
0,1000025,2,0,benign,0.0,0,benign,1
1,1002945,2,0,benign,1.0,1,malignant,0
2,1015425,2,0,benign,0.0,0,benign,1
3,1016277,2,0,benign,1.0,1,malignant,0
4,1017023,2,0,benign,0.0,0,benign,1



5 mau du doan sai:


,sample_code_number,class,target,target_label,y_pred_raw,y_pred,pred_label,correct
1,1002945,2,0,benign,1.0,1,malignant,0
3,1016277,2,0,benign,1.0,1,malignant,0
12,1041801,4,1,malignant,0.0,0,benign,0
25,1065726,4,1,malignant,0.0,0,benign,0
39,1091262,4,1,malignant,0.0,0,benign,0


In [26]:
# Doi chieu voi tap test/check da luu luc train (split paper 200/263/200)
from sklearn.model_selection import train_test_split

# Tai tao split giong notebook train: outlier -> quality drop -> stratified split
outlier_std_multiplier = meta.get("outlier_std_multiplier", 2.5)
data_center = X_raw.mean(axis=0)
euclidean_distances = np.linalg.norm(X_raw - data_center, axis=1)
outlier_threshold = euclidean_distances.mean() + outlier_std_multiplier * euclidean_distances.std()
outlier_mask = euclidean_distances > outlier_threshold
keep_mask = ~outlier_mask

X_clean = X_raw[keep_mask]
y_clean = y_true[keep_mask]
X_norm_clean = scaler.transform(X_clean).astype(np.float32)
selected_idx_clean = [feature_cols.index(c) for c in selected_feature_names]
X_selected = X_norm_clean[:, selected_idx_clean].astype(np.float32)

n_quality_drop = len(X_selected) - meta["used_samples"]
centroid_benign = X_selected[y_clean == 0].mean(axis=0)
centroid_malignant = X_selected[y_clean == 1].mean(axis=0)
dist_to_benign = np.linalg.norm(X_selected - centroid_benign, axis=1)
dist_to_malignant = np.linalg.norm(X_selected - centroid_malignant, axis=1)
own_dist = np.where(y_clean == 0, dist_to_benign, dist_to_malignant)
other_dist = np.where(y_clean == 0, dist_to_malignant, dist_to_benign)
quality_score = other_dist - own_dist

drop_idx = np.argsort(quality_score)[:n_quality_drop]
quality_keep_mask = np.ones(len(X_selected), dtype=bool)
quality_keep_mask[drop_idx] = False

X_663 = X_selected[quality_keep_mask]
y_663 = y_clean[quality_keep_mask]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_663, y_663, train_size=meta["train_size"], random_state=42, stratify=y_663,
)
X_check, X_test, y_check, y_test = train_test_split(
    X_temp, y_temp, train_size=meta["check_size"], random_state=42, stratify=y_temp,
)

y_check_pred = to_binary(model.predict(X_check))
y_test_pred = to_binary(model.predict(X_test))

check_metrics = evaluate_split("CHECK", y_check, y_check_pred)
test_metrics = evaluate_split("TEST", y_test, y_test_pred)

compare_df = pd.DataFrame([
    {"split": "CHECK", "saved_accuracy": saved_metrics["metrics"]["CHECK"]["accuracy"], "inference_accuracy": check_metrics["accuracy"]},
    {"split": "TEST", "saved_accuracy": saved_metrics["metrics"]["TEST"]["accuracy"], "inference_accuracy": test_metrics["accuracy"]},
])

print("Doi chieu accuracy voi metrics da luu luc train:")
display(compare_df)

quality_dropped_codes = set(meta.get("quality_dropped_sample_code_numbers", []))
outlier_drop_csv_path = models_dir / f"{STAMP}_paperstyle_outlier_dropped_samples.csv"
outlier_dropped_codes = set()
if outlier_drop_csv_path.exists():
    outlier_dropped_codes = set(pd.read_csv(outlier_drop_csv_path)["sample_code_number"].astype(int))

result_df["paper_split_group"] = "used_663"
result_df.loc[result_df["sample_code_number"].isin(quality_dropped_codes), "paper_split_group"] = "quality_dropped"
result_df.loc[result_df["sample_code_number"].isin(outlier_dropped_codes), "paper_split_group"] = "outlier_dropped"

group_metrics = []
for group_name, mask in [
    ("used_663", result_df["paper_split_group"] == "used_663"),
    ("quality_dropped", result_df["paper_split_group"] == "quality_dropped"),
    ("outlier_dropped", result_df["paper_split_group"] == "outlier_dropped"),
]:
    sub = result_df[mask]
    if len(sub) == 0:
        continue
    group_metrics.append(evaluate_split(group_name, sub["target"].values, sub["y_pred"].values))

print("\nMetrics theo nhom paper split:")
display(pd.DataFrame(group_metrics))

Doi chieu accuracy voi metrics da luu luc train:


,split,saved_accuracy,inference_accuracy
0,CHECK,0.942966,0.942966
1,TEST,0.935000,0.935000



Metrics theo nhom paper split:


,split,accuracy,precision,recall,f1_score,roc_auc,tn,fp,fn,tp,support
0,used_663,0.953030,0.923423,0.936073,0.929705,0.948762,424,17,14,205,660
1,quality_dropped,0.333333,0.000000,0.000000,0.000000,NaN,1,2,0,0,3
2,outlier_dropped,1.000000,1.000000,1.000000,1.000000,NaN,0,0,0,20,20


In [27]:
# Luu ket qua predict
out_csv = models_dir / f"{STAMP}_full_wbcd_predictions.csv"
result_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print("Tong mau:", len(result_df), "| Dung:", int(result_df["correct"].sum()), "| Sai:", int((result_df["correct"] == 0).sum()))

Saved: models\20260621_174830_full_wbcd_predictions.csv
Tong mau: 683 | Dung: 650 | Sai: 33
